# 量子化で LLM の性能がどれだけ落ちるかを測る（bitsandbytes と GGUF）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/23_llm_analysis/quantization_eval.ipynb)

NLP クックブック 23-04 ／ LLM の評価と分析

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

同じ LLM を 16bit、bitsandbytes の 8bit と 4bit (NF4)、GGUF（Q8_0、Q4_K_M、Q2_K）で動かし、パープレキシティ、選択式のベンチマークの正解率、メモリ、生成の速さを 1 つの表で比べたい。差がばらつきの範囲かどうかも確かめ、「大きいモデルの 4bit」と「小さいモデルの 16bit」のどちらがよいかも見る。

In [1]:
# llama-cpp-python は CUDA 13 用のビルド済みのパッケージを入れる（ソースからのビルドは不要）。0.3.37 の CPU 部分は AVX-512 を前提にビルドされていて動かない CPU があるので 0.3.36 に固定する
!pip install -q datasets bitsandbytes==0.50.2 llama-cpp-python==0.3.36 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130
# GGUF への変換スクリプト（llama.cpp、MIT ライセンス）。llama-cpp-python 0.3.36 が同梱する llama.cpp と同じ版 (b11312)
!git -c advice.detachedHead=false clone -q --depth 1 --branch b11312 https://github.com/ggml-org/llama.cpp

In [2]:
import ctypes, gc, logging, os, sys, time
from collections import Counter
import llama_cpp
import numpy as np
import pandas as pd
import pynvml
import sentencepiece as spm
import torch
from datasets import load_dataset
from huggingface_hub import hf_hub_download, snapshot_download
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

torch.manual_seed(0)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
DTYPE = torch.bfloat16 if BF16 else torch.float16
NAMES = {s: f"sbintuitions/sarashina2.2-{s}-instruct-v0.1" for s in ["0.5b", "1b", "3b"]}   # SB Intuitions、MIT ライセンス

# JGLUE (Kurihara et al., 2022) の JCommonsenseQA（5 択）。CC BY-SA 4.0。この章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」と同じ 500 問
jcqa = load_dataset("sbintuitions/JCommonsenseQA", split="validation").shuffle(seed=0).select(range(500))
gold = np.array(jcqa["label"])
# パープレキシティを測る文章: JSQuAD (CC BY-SA 4.0) の評価データの段落（Wikipedia の記事）から 100 個
paras = list(dict.fromkeys(c.split("[SEP]")[-1].strip() for c in load_dataset("sbintuitions/JSQuAD", split="validation")["context"]))
texts = [paras[i] for i in np.random.default_rng(0).choice(len(paras), 100, replace=False)]

tokenizer = AutoTokenizer.from_pretrained(NAMES["3b"])      # 3 つのモデルは同じ tokenizer.model を使う
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す（特殊トークンやチャットテンプレートはそのまま）
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(NAMES["3b"], "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)

def chat(text):
    return tokenizer.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True, tokenize=False)

def mc_prompt(ex):      # この章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」と同じ 0-shot の指示
    listing = "\n".join(f"{i + 1}. {ex[f'choice{i}']}" for i in range(5))
    return f"次の質問の答えとして最も適切なものを選び、番号だけを答えてください。\n\n質問: {ex['question']}\n{listing}"

# どの形式のモデルにも、同じトークン列を渡す（GGUF でも llama.cpp のトークナイザーは使わない）
jcqa_ids = [tokenizer(chat(mc_prompt(ex)), add_special_tokens=False)["input_ids"] for ex in jcqa]
ppl_ids = [[tokenizer.bos_token_id] + tokenizer(t, add_special_tokens=False)["input_ids"][:511] for t in texts]
digits = [tokenizer(str(i), add_special_tokens=False)["input_ids"][-1] for i in range(1, 6)]
gen_ids = tokenizer(chat("日本の四季について、それぞれの特徴を詳しく説明してください。"), add_special_tokens=False)["input_ids"]
print(f"JCQA {len(jcqa_ids)} 問（平均 {np.mean([len(s) for s in jcqa_ids]):.0f} トークン）、段落 {len(ppl_ids)} 個（{sum(len(s) - 1 for s in ppl_ids)} トークン）")

JCQA 500 問（平均 56 トークン）、段落 100 個（8409 トークン）


In [3]:
# 3b を GGUF (16bit) に変換し、llama.cpp の量子化で Q8_0、Q4_K_M、Q2_K を作る（Colab で 3b のダウンロード約 7GB を含めて 10 分ほど）
local = snapshot_download(NAMES["3b"])
!python llama.cpp/convert_hf_to_gguf.py {local} --outtype f16 --outfile 3b-F16.gguf > convert.log 2>&1
print(f"F16: {os.path.getsize('3b-F16.gguf') / 1e9:.2f} GB")
logging.getLogger("llama-cpp-python").setLevel(logging.ERROR)     # llama.cpp の詳しいログを出さない
FTYPES = {"Q8_0": llama_cpp.LLAMA_FTYPE_MOSTLY_Q8_0, "Q4_K_M": llama_cpp.LLAMA_FTYPE_MOSTLY_Q4_K_M, "Q2_K": llama_cpp.LLAMA_FTYPE_MOSTLY_Q2_K}
for q, ftype in FTYPES.items():
    t = time.time()
    params = llama_cpp.llama_model_quantize_default_params()
    params.ftype, params.nthread = ftype, os.cpu_count()
    assert llama_cpp.llama_model_quantize(b"3b-F16.gguf", f"3b-{q}.gguf".encode(), ctypes.byref(params)) == 0
    print(f"{q}: {os.path.getsize(f'3b-{q}.gguf') / 1e9:.2f} GB（{time.time() - t:.0f} 秒）")

# Q4_K_M の中身: テンソルごとの量子化の型を数える（クローンした llama.cpp に付属する gguf-py で GGUF を読む）
sys.path.insert(0, "llama.cpp/gguf-py")
from gguf import GGUFReader
tensors = GGUFReader("3b-Q4_K_M.gguf").tensors
print("Q4_K_M のテンソルの型:", dict(Counter(t.tensor_type.name for t in tensors)))
print("埋め込みと出力層:", {t.name: t.tensor_type.name for t in tensors if t.name in ("token_embd.weight", "output.weight")})

F16: 6.71 GB
Q8_0: 3.57 GB（19 秒）
Q4_K_M: 2.07 GB（169 秒）
Q2_K: 1.34 GB（114 秒）
Q4_K_M のテンソルの型: {'Q6_K': 33, 'F32': 65, 'Q4_K': 193}
埋め込みと出力層: {'output.weight': 'Q6_K', 'token_embd.weight': 'Q4_K'}


In [4]:
pynvml.nvmlInit()
_dev = pynvml.nvmlDeviceGetHandleByUUID("GPU-" + str(torch.cuda.get_device_properties(0).uuid))
def gpu_used():
    """このプロセスが GPU で使っているメモリ (bytes)。llama.cpp が確保した分は PyTorch から見えないので NVML で測る"""
    procs = {p.pid: p.usedGpuMemory for p in pynvml.nvmlDeviceGetComputeRunningProcesses(_dev)}
    if procs.get(os.getpid()):
        return procs[os.getpid()]
    free, total = torch.cuda.mem_get_info()     # プロセスごとの値が取れない環境では、GPU 全体の使用量で代用する
    return total - free

@torch.no_grad()
def eval_hf(model):
    """Transformers のモデルで、(1) JCQA の番号「1」〜「5」の対数確率、(2) 段落ごとの負の対数尤度の合計、(3) 生成の速さを求める"""
    lps = []
    for i in range(0, len(jcqa_ids), 32):
        b = jcqa_ids[i:i + 32]
        L = max(map(len, b))
        ids = torch.tensor([[tokenizer.pad_token_id] * (L - len(s)) + s for s in b], device=model.device)     # 左パディング
        mask = (torch.arange(L, device=model.device) >= torch.tensor([L - len(s) for s in b], device=model.device)[:, None]).long()
        pos = (mask.cumsum(-1) - 1).clamp(min=0)
        logits = model(input_ids=ids, attention_mask=mask, position_ids=pos, logits_to_keep=1).logits[:, -1, digits]
        lps.append(logits.float().log_softmax(-1).cpu().numpy())
    nll = []
    for s in ppl_ids:
        ids = torch.tensor([s], device=model.device)
        nll.append(-model(ids).logits[0, :-1].float().log_softmax(-1).gather(1, ids[0, 1:, None]).sum().item())
    ids = torch.tensor([gen_ids], device=model.device)
    kw = dict(attention_mask=torch.ones_like(ids), do_sample=False, pad_token_id=tokenizer.pad_token_id)
    model.generate(ids, max_new_tokens=8, **kw)                     # 1 回目は準備の時間がかかるので、測る前に一度動かす
    torch.cuda.synchronize(); t = time.time()
    out = model.generate(ids, max_new_tokens=128, min_new_tokens=128, **kw)
    torch.cuda.synchronize()
    return np.concatenate(lps), np.array(nll), (out.shape[1] - ids.shape[1]) / (time.time() - t)

def eval_gguf_ids(llm, mc_ids, text_ids):
    """llama.cpp が返すロジットから、番号の対数確率と段落の負の対数尤度を自分で計算する"""
    lps, nll = [], []
    for s in mc_ids:
        llm.reset(); llm.eval(s)
        lps.append(torch.from_numpy(llm.scores[len(s) - 1, digits]).log_softmax(-1).numpy())
    for s in text_ids:
        llm.reset(); llm.eval(s)
        lp = torch.from_numpy(llm.scores[:len(s) - 1]).log_softmax(-1).gather(1, torch.tensor(s[1:])[:, None])
        nll.append(-lp.sum().item())
    return np.stack(lps), np.array(nll)

def eval_gguf(llm):
    """GGUF のモデルで同じ 3 つを求める（Transformers と同じトークン列を渡す）"""
    lps, nll = eval_gguf_ids(llm, jcqa_ids, ppl_ids)
    kw = dict(temperature=0.0, logit_bias={tokenizer.eos_token_id: float("-inf")})    # 貪欲法、128 トークンまで止めない
    llm.reset(); llm.create_completion(gen_ids, max_tokens=8, **kw)
    llm.reset(); t = time.time()                                    # reset しないとプロンプトの計算結果が再利用される
    out = llm.create_completion(gen_ids, max_tokens=128, **kw)
    return lps, nll, out["usage"]["completion_tokens"] / (time.time() - t)

In [5]:
# 10 通りを順に評価する（Colab の T4 で 5〜10 分。1b と 0.5b のダウンロード約 4.5GB の時間がこれに加わる）
# 量子化しない層（埋め込み、出力層、正規化）も 16bit にそろえるため、bitsandbytes でも dtype を指定する
nf4 = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True)
CONFIGS = {   # 名前:（モデル, Transformers の読み込みの引数 または GGUF のファイル）
    "3b 16bit": ("3b", {"dtype": DTYPE}),
    "3b bnb 8bit": ("3b", {"dtype": DTYPE, "quantization_config": BitsAndBytesConfig(load_in_8bit=True)}),
    "3b bnb 4bit (NF4)": ("3b", {"dtype": DTYPE, "quantization_config": nf4}),     # 大規模言語モデルの章の「LLM を 4bit 量子化して少ないメモリで動かす」と同じ量子化の設定
    "3b GGUF F16": ("3b", "3b-F16.gguf"),
    "3b GGUF Q8_0": ("3b", "3b-Q8_0.gguf"),
    "3b GGUF Q4_K_M": ("3b", "3b-Q4_K_M.gguf"),
    "3b GGUF Q2_K": ("3b", "3b-Q2_K.gguf"),
    "1b 16bit": ("1b", {"dtype": DTYPE}),
    "1b bnb 4bit (NF4)": ("1b", {"dtype": DTYPE, "quantization_config": nf4}),
    "0.5b 16bit": ("0.5b", {"dtype": DTYPE}),
}
res = {}
for key, (size, spec) in CONFIGS.items():
    gc.collect(); torch.cuda.empty_cache(); base = gpu_used(); t = time.time()
    if isinstance(spec, dict):
        model = AutoModelForCausalLM.from_pretrained(NAMES[size], device_map="cuda", **spec).eval()
        weights = model.get_memory_footprint()
        lp, nll, speed = eval_hf(model)
    else:
        model = llama_cpp.Llama(spec, n_gpu_layers=-1, n_ctx=1024, n_batch=512, logits_all=True, verbose=False)   # 全層を GPU に載せる
        weights = os.path.getsize(spec)
        lp, nll, speed = eval_gguf(model)
    res[key] = dict(lp=lp, nll=nll, speed=speed, weights=weights / 1e9, gpu=(gpu_used() - base) / 1e9)
    print(f"{key}: {time.time() - t:.0f} 秒")
    if not isinstance(spec, dict):
        model.close()
    del model
print(f"PyTorch の GPU メモリの最大: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

3b 16bit: 15 秒
3b bnb 8bit: 38 秒
3b bnb 4bit (NF4): 16 秒
3b GGUF F16: 17 秒
3b GGUF Q8_0: 15 秒
3b GGUF Q4_K_M: 14 秒
3b GGUF Q2_K: 16 秒
1b 16bit: 13 秒
1b bnb 4bit (NF4): 11 秒
0.5b 16bit: 7 秒
PyTorch の GPU メモリの最大: 6.9 GB


In [6]:
# 1 つの表にまとめる。[ ] は 95% 信頼区間（問題・段落を復元抽出するブートストラップ。差と比は同じ問題・段落で比べる対応のある方法）
rng = np.random.default_rng(0)
B, Bp = rng.integers(0, len(gold), (2000, len(gold))), rng.integers(0, len(texts), (2000, len(texts)))
ntok = np.array([len(s) - 1 for s in ppl_ids])
ppl = lambda nll, idx=slice(None): np.exp(nll[idx].sum(-1) / ntok[idx].sum(-1))
acc = {k: (r["lp"].argmax(1) == gold).astype(float) for k, r in res.items()}

def compare(a, b):
    """a と b の正解率の差（ポイント）と、パープレキシティの増加 (%)"""
    d = (acc[a][B].mean(1) - acc[b][B].mean(1)) * 100
    ratio = (ppl(res[a]["nll"], Bp) / ppl(res[b]["nll"], Bp) - 1) * 100
    return (f"{(acc[a].mean() - acc[b].mean()) * 100:+.1f} [{np.percentile(d, 2.5):+.1f}, {np.percentile(d, 97.5):+.1f}]",
            f"{(ppl(res[a]['nll']) / ppl(res[b]['nll']) - 1) * 100:+.1f} [{np.percentile(ratio, 2.5):+.1f}, {np.percentile(ratio, 97.5):+.1f}]")

rows = {}
for k, r in res.items():
    d_acc, d_ppl = compare(k, "3b 16bit")
    rows[k] = {"JCQA 正解率 (%)": f"{acc[k].mean() * 100:.1f} [{np.percentile(acc[k][B].mean(1), 2.5) * 100:.1f}, {np.percentile(acc[k][B].mean(1), 97.5) * 100:.1f}]", "3b 16bit との差": d_acc,
               "答えが変わった (%)": f"{(r['lp'].argmax(1) != res['3b 16bit']['lp'].argmax(1)).mean() * 100:.1f}",
               "PPL": f"{ppl(r['nll']):.2f}", "PPL の増加 (%)": d_ppl,
               "重み (GB)": f"{r['weights']:.2f}", "GPU 使用量 (GB)": f"{r['gpu']:.2f}", "生成（トークン/秒）": f"{r['speed']:.1f}"}
pd.set_option("display.width", 250)
print(pd.DataFrame(rows).T.to_string())
print(f"（JCQA {len(gold)} 問、偶然の正解率 20%。PPL は段落 {len(texts)} 個・{ntok.sum()} トークン。生成は 1 文ずつ 128 トークン）")

# 「大きいモデルの 4bit」と「小さいモデルの 16bit」を直接比べる
for a, b in [("3b bnb 4bit (NF4)", "1b 16bit"), ("3b GGUF Q4_K_M", "1b 16bit"), ("1b bnb 4bit (NF4)", "1b 16bit")]:
    d_acc, d_ppl = compare(a, b)
    print(f"{a} − {b}: 正解率 {d_acc} ポイント、PPL の増加 {d_ppl} %、重み {res[a]['weights']:.2f} GB / {res[b]['weights']:.2f} GB")

                        JCQA 正解率 (%)          3b 16bit との差 答えが変わった (%)    PPL           PPL の増加 (%) 重み (GB) GPU 使用量 (GB) 生成（トークン/秒）
3b 16bit           90.4 [87.8, 92.8]     +0.0 [+0.0, +0.0]         0.0  24.98     +0.0 [+0.0, +0.0]    6.71         8.06       45.7
3b bnb 8bit        90.4 [87.8, 93.0]     +0.0 [-1.0, +1.0]         1.4  24.94     -0.2 [-0.7, +0.3]    3.88         4.54       10.6
3b bnb 4bit (NF4)  89.8 [87.0, 92.4]     -0.6 [-2.2, +1.2]         3.8  24.38     -2.4 [-3.9, -0.9]    2.46         3.59       35.2
3b GGUF F16        90.4 [87.8, 92.8]     +0.0 [+0.0, +0.0]         0.0  24.99     +0.1 [-0.0, +0.1]    6.71         6.64       93.4
3b GGUF Q8_0       90.4 [87.8, 92.8]     +0.0 [-0.6, +0.6]         0.4  25.20     +0.9 [+0.7, +1.1]    3.57         3.69      143.0
3b GGUF Q4_K_M     91.0 [88.4, 93.4]     +0.6 [-0.8, +2.0]         3.0  27.32    +9.4 [+7.9, +11.1]    2.07         2.32      186.2
3b GGUF Q2_K       85.2 [82.2, 88.4]     -5.2 [-8.6, -1.6]        15.8  37.8

In [7]:
# llama-cpp-python に文字列を渡すと llama.cpp のトークナイザーで分割される。その違いが結果に響くかを Q4_K_M で確かめる
llm = llama_cpp.Llama("3b-Q4_K_M.gguf", n_gpu_layers=-1, n_ctx=1024, n_batch=512, logits_all=True, verbose=False)
own = lambda text: llm.tokenize(text.encode(), add_bos=False, special=True)
own_jcqa = [own(chat(mc_prompt(ex))) for ex in jcqa]
own_ppl = [[tokenizer.bos_token_id] + own(t) for t in texts]
assert max(map(len, ppl_ids + own_ppl)) < 512                   # 段落を途中で切っていない（文字あたりで比べられる）
print("分割が一致した JCQA のプロンプト:", sum(a == b for a, b in zip(own_jcqa, jcqa_ids)), "/", len(jcqa),
      "/ 段落:", sum(a == b for a, b in zip(own_ppl, ppl_ids)), "/", len(texts))
for a, b in zip(ppl_ids, own_ppl):      # 最初に食い違う箇所を 1 つ見せる
    k = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), None)
    if k is not None:
        print("学習時と同じ分割:", tokenizer.convert_ids_to_tokens(a[k:k + 4]), "/ llama.cpp:", tokenizer.convert_ids_to_tokens(b[k:k + 4]))
        break
r = res["3b GGUF Q4_K_M"]
lp_own, nll_own = eval_gguf_ids(llm, own_jcqa, own_ppl)
acc_own, acc_hf = (lp_own.argmax(1) == gold).astype(float), (r["lp"].argmax(1) == gold).astype(float)
d = (acc_own[B].mean(1) - acc_hf[B].mean(1)) * 100
print(f"JCQA 正解率: 学習時と同じ分割 {acc_hf.mean():.1%} / llama.cpp の分割 {acc_own.mean():.1%}（差 {(acc_own.mean() - acc_hf.mean()) * 100:+.1f} [{np.percentile(d, 2.5):+.1f}, {np.percentile(d, 97.5):+.1f}] ポイント）")
nchar = sum(map(len, texts))       # トークン数が違うので、1 文字あたりの負の対数尤度で比べる
print(f"1 文字あたりの負の対数尤度: 学習時と同じ分割 {r['nll'].sum() / nchar:.3f} / llama.cpp の分割 {nll_own.sum() / nchar:.3f}"
      f"（トークン数 {sum(len(s) - 1 for s in ppl_ids)} / {sum(len(s) - 1 for s in own_ppl)}）")
llm.close()

分割が一致した JCQA のプロンプト: 0 / 500 / 段落: 1 / 100
学習時と同じ分割: ['これ', 'を通じて', '行き来', 'できる'] / llama.cpp: ['これを', '通じ', 'て', '行き来']
JCQA 正解率: 学習時と同じ分割 91.0% / llama.cpp の分割 90.8%（差 -0.2 [-1.4, +1.0] ポイント）
1 文字あたりの負の対数尤度: 学習時と同じ分割 1.640 / llama.cpp の分割 1.990（トークン数 8409 / 8999）


- 3b では、8bit（bitsandbytes の LLM.int8()、GGUF の Q8_0）は 16bit とほとんど変わらなかった（JCommonsenseQA の差 ±0.0 ポイント、パープレキシティの増加 −0.2% と +0.9%）。4bit（NF4、Q4_K_M）も正解率の差は −0.6 [−2.2, +1.2] と +0.6 [−0.8, +2.0] ポイントで、信頼区間が 0 をまたいだ。2bit の Q2_K は −5.2 [−8.6, −1.6] ポイント、パープレキシティ +51.6% と、はっきり落ちた。大規模言語モデルの章の「LLM を 4bit 量子化して少ないメモリで動かす」の「精度の低下はわずか」は、このモデルとタスクでは 4bit まで当てはまり、2bit では当てはまらない。ただし 500 問では 2 ポイント程度の差は見分けられないので、「差がない」とまでは言えない。
- 正解率だけを見ると量子化の影響を見落とす。Q4_K_M は正解率が変わらなくても答えの 3.0% が入れ替わり、パープレキシティは +9.4% [+7.9, +11.1] 増えた。一方、3b の NF4 はパープレキシティが −2.4% [−3.9, −0.9] と、むしろ下がった（1b の NF4 は 1b の 16bit より +4.5% [+3.2, +5.7]）。量子化で必ず一様に悪くなるわけではなく、原因はここでは確かめていない。bitsandbytes は埋め込みと出力層を量子化せず 16bit のまま残し（重みの大きさが 2.46GB と Q4_K_M の 2.07GB より大きいのはそのため）、Q4_K_M は埋め込みも 4bit にし、出力層などを 6bit にするので、同じ「4bit」でも中身が違う。自分のタスクで、正解率、答えの入れ替わり、パープレキシティを合わせて確かめる。
- 「大きいモデルの 4bit」は「小さいモデルの 16bit」よりよかった。3b の NF4 は 1b の 16bit より正解率が +5.0 [+1.4, +8.4] ポイント高く、パープレキシティは 42.1% 低く、重みは 2.46GB と 2.82GB で小さい。3b の Q4_K_M (2.07GB) でも +6.2 [+2.6, +9.6] ポイントだった。メモリが限られるなら、小さいモデルを 16bit で使うより、大きいモデルを 4bit にするほうがよい（このモデルの系列とこの 2 つの指標での結果）。
- 条件をそろえるために、GGUF でも Transformers と同じトークン列を渡し、llama.cpp が返すロジットから対数確率を自分で計算した。GGUF の F16 は Transformers の 16bit と正解率が同じで、パープレキシティの違いも +0.1% だったので、計算の方法による差はほぼない。llama-cpp-python に文字列を渡すと llama.cpp のトークナイザーで分割され、このモデル（SentencePiece の Unigram）では JCommonsenseQA のプロンプト 500 個がすべて、段落は 100 個中 99 個が学習時と違う分割になった。正解率は −0.2 [−1.4, +1.0] ポイントで変わらなかったが、1 文字あたりの負の対数尤度は 1.640 から 1.990 に悪化した。GGUF の評価値を他の形式と比べるときは、分割がそろっているかを確かめる。
- メモリと速さ: 重みの大きさは 16bit の 6.71GB に対し、Q8_0 で 3.57GB、Q4_K_M で 2.07GB、Q2_K で 1.34GB になった。GPU 使用量は NVML で測ったこのプロセスの使用量の増分で、活性化や KV キャッシュを含み、バッチの大きさで変わる。最大は 3b の 16bit の 8.06GB で、T4 の 15GB に収まる（PyTorch の GPU メモリの最大は 6.9GB だが、llama.cpp が確保した分はこれに入らない）。プロセスごとの値が取れない環境では GPU 全体の使用量で代用するので、ほかのプロセスがあると不正確になる。1 文ずつの生成の速さ（トークン/秒）は、bitsandbytes では 16bit の 45.7 より遅くなった（8bit で 10.6、NF4 で 35.2）。llama.cpp の中では F16 の 93.4 に対し、Q8_0 で 143.0、Q4_K_M で 186.2 と速くなった。Transformers と llama.cpp の間には量子化しない 16bit どうしでも約 2 倍の差があり、これは量子化ではなく実行するソフトの違いによる。速さは検証に使った GPU での値で、T4 では違い、GPU をほかの処理と共有していると実行のたびに変わる。形式の間の大小だけを参考にする。
- GGUF は配布されたものを使わず、同じ重みから llama.cpp (b11312) の変換スクリプトと llama-cpp-python の `llama_model_quantize` で作った（Hugging Face Hub にはほかの人が変換した GGUF もあるが、変換の版や設定がそろわない）。Q4_K_M は重みを 4bit の Q4_K にし、出力層を含む一部を 6bit の Q6_K にする（量子化のセルの出力では Q4_K が 193 個、Q6_K が 33 個、正規化の重みなど F32 のままのものが 65 個）。重要度行列 (imatrix) を使う量子化は、特に Q2_K のような低いビット数で劣化を減らすとされるが、ここでは試していない。llama-cpp-python は CUDA 13 用のビルド済みパッケージを入れる（ソースからのビルドは不要。Colab の torch は CUDA 13 版）。Colab では、3 つのモデルのダウンロード（合わせて約 11GB）、CPU での変換と量子化、評価を合わせて 20〜25 分ほどかかる見込みである。
- モデルは SB Intuitions の sarashina2.2（MIT ライセンス）、データは JGLUE の JCommonsenseQA と JSQuAD (CC BY-SA 4.0)、llama.cpp と llama-cpp-python と bitsandbytes は MIT ライセンス。パープレキシティを測った段落は Wikipedia の記事なので、学習データに含まれていた可能性があり、値そのものより形式の間の比を見る。評価の方法は、この章の「日本語のベンチマークで複数の LLM を同じ条件で比べる」と同じ（番号の確率、0-shot）。